# Step 4B.1 — H4 перевірка: score-level rank fusion B1 (EPSS) + B3 (SecBERT text)
Найдешевший тест гіпотези H4 ("текст несе комплементарний EPSS сигнал") — БЕЗ навчання
нової складної моделі. Просто зважене ранжування:

S_fusion = α · rank(EPSS) + (1-α) · rank(B3)

α обирається ЛИШЕ на validation (малий грід), test — одна фінальна оцінка.
Якщо fusion AP > B1 AP на test — сигнал комплементарності підтверджено, і є підстава
переходити до Кроку 2 (B4-A/B4-B). Якщо ні — H4 не підтверджується цим тестом.

B1 (raw EPSS) береться напряму. B3-classifier перенавчається із frozen train split за
ІДЕНТИЧНИМ протоколом Step 4A.10 (той самий split, ознаки, C-грід, validation selection),
щоб відтворити його зафіксовані validation/test скори — жодних нових даних, ознак, спліту
чи гіперпараметрів не вводиться. Точність відтворення перевіряється явним assert проти
зафіксованих у Step 4A.10 test AP=0.005151 / ROC-AUC=0.775979.

`rank(.)` — нормалізований percentile/mid-rank (`scipy.stats.rankdata`, однакові значення
отримують середній ранг), обчислений НЕЗАЛЕЖНО в межах кожного спліту (val окремо, test окремо).

Ядро: IDS Research.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.stats import rankdata

CTI_DIR = Path("/srv/ids_research/data/cti")
RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"

HORIZON = 180
TARGET_COL = f"future_kev_{HORIZON}d"
SPLIT_COL = f"split_{HORIZON}d"
TOP_K_FRACTIONS = [0.001, 0.005, 0.01, 0.05]
EXPECTED_SPLIT_SIZES = {"train": 249725, "val": 15192, "test": 65159}
ALPHA_GRID = np.arange(0.0, 1.01, 0.1)   # 0.0=чистий B3, 1.0=чистий B1; обираємо ЛИШЕ на val

print(f"HORIZON={HORIZON}, ALPHA_GRID={np.round(ALPHA_GRID, 2).tolist()}")

HORIZON=180, ALPHA_GRID=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


In [2]:
# ---------- Відтворення B1 (raw EPSS) і B3-classifier за ІДЕНТИЧНИМ Step 4A.10 протоколом ----------
# B1 = сирий epss_t напряму (жодного навчання, як у Step 4A.4).
# B3-classifier ПЕРЕНАВЧАЄТЬСЯ тут із frozen train split, оскільки сам fitted-об'єкт зі
# Step 4A.10 не зберігався — але СПЛІТ, ОЗНАКИ, C-грід і validation selection ІДЕНТИЧНІ,
# тож відтворені скори мають збігатися з уже зафіксованими. Точність відтворення
# перевіряється явним assert нижче на val, і ОКРЕМО на test (найважливіше — саме test,
# бо B4 залежатиме від точного відтворення B3).
ds = pd.read_parquet(CTI_DIR / "cve_forecast_dataset.parquet",
                     columns=["cve_id", TARGET_COL, SPLIT_COL, "epss_t"])
observed = ds[ds[TARGET_COL].notna()].copy()
observed[TARGET_COL] = observed[TARGET_COL].astype(int)

for split_name, expected in EXPECTED_SPLIT_SIZES.items():
    actual = len(observed[observed[SPLIT_COL] == split_name])
    assert actual == expected, f"{split_name}: actual={actual}, expected={expected} — dataset змінився!"
print("✅ Розміри train/val/test збігаються із зафіксованим Step 4A.2")

train_df = observed[observed[SPLIT_COL] == "train"].copy()
val_df = observed[observed[SPLIT_COL] == "val"].copy()
test_df = observed[observed[SPLIT_COL] == "test"].copy()

# B1: сирий EPSS
b1_val = val_df["epss_t"].values
b1_test = test_df["epss_t"].values

# B3: точне відтворення Step 4A.10 (manual weighting, C обирається так само на validation)
embeddings = np.load(RESULTS_DIR / "step4a9_b3_embeddings.npy")
cve_id_order = pd.read_parquet(RESULTS_DIR / "step4a9_b3_cve_id_mapping.parquet")["cve_id"]
cve_id_to_row = pd.Series(np.arange(len(cve_id_order)), index=cve_id_order.values)

def build_text_matrix(df):
    return embeddings[cve_id_to_row.loc[df["cve_id"]].values]

X_train_txt, X_val_txt, X_test_txt = build_text_matrix(train_df), build_text_matrix(val_df), build_text_matrix(test_df)
y_train, y_val, y_test = train_df[TARGET_COL].values, val_df[TARGET_COL].values, test_df[TARGET_COL].values

from sklearn.linear_model import LogisticRegression
n_pos_train = int(y_train.sum())
n_neg_train = len(y_train) - n_pos_train
manual_weight = {0: 1.0, 1: float(n_neg_train / n_pos_train)}

b3_c_search = []
best_c_b3, best_val_ap_b3, b3_model = None, -1, None
for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    clf = LogisticRegression(C=C, class_weight=manual_weight, max_iter=1000, random_state=42, solver="lbfgs")
    clf.fit(X_train_txt, y_train)
    val_scores = clf.predict_proba(X_val_txt)[:, 1]
    ap = average_precision_score(y_val, val_scores)
    b3_c_search.append((C, ap))
    if ap > best_val_ap_b3:
        best_c_b3, best_val_ap_b3, b3_model = C, ap, clf

print(f"B3 (відтворено): best_C={best_c_b3}, val_AP={best_val_ap_b3:.6f} "
      f"(звірте зі Step 4A.10: очікується C=0.1, val_AP≈0.009281)")
assert abs(best_val_ap_b3 - 0.009281) < 0.001, \
    "Відтворений B3 val_AP суттєво відрізняється від зафіксованого Step 4A.10 — перевірте!"
print("✅ Відтворений B3 val_AP узгоджується із зафіксованим Step 4A.10 результатом")

b3_val = b3_model.predict_proba(X_val_txt)[:, 1]
b3_test = b3_model.predict_proba(X_test_txt)[:, 1]

# ОБОВ'ЯЗКОВА звірка на TEST — найважливіша перевірка, бо весь висновок H4 спирається на
# точне відтворення B3. Якщо це впаде — далі fusion-результат НЕ можна довіряти.
b3_ap_test_reproduced = average_precision_score(y_test, b3_test)
b3_roc_test_reproduced = roc_auc_score(y_test, b3_test)
print(f"\nB3 reproduced test: AP={b3_ap_test_reproduced:.6f}, ROC-AUC={b3_roc_test_reproduced:.4f}")
assert abs(b3_ap_test_reproduced - 0.005151) < 0.0005, \
    f"Відтворений B3 test AP={b3_ap_test_reproduced:.6f} суттєво відрізняється від зафіксованого 0.005151!"
assert abs(b3_roc_test_reproduced - 0.775979) < 0.001, \
    f"Відтворений B3 test ROC-AUC={b3_roc_test_reproduced:.4f} суттєво відрізняється від зафіксованого 0.775979!"
print("✅ Відтворений B3 TEST результат узгоджується із зафіксованим Step 4A.10")

✅ Розміри train/val/test збігаються із зафіксованим Step 4A.2
B3 (відтворено): best_C=0.1, val_AP=0.009281 (звірте зі Step 4A.10: очікується C=0.1, val_AP≈0.009281)
✅ Відтворений B3 val_AP узгоджується із зафіксованим Step 4A.10 результатом

B3 reproduced test: AP=0.005151, ROC-AUC=0.7760
✅ Відтворений B3 TEST результат узгоджується із зафіксованим Step 4A.10


In [3]:
# ---------- Rank fusion: S_fusion = α·rank(EPSS) + (1-α)·rank(B3), α обирається ЛИШЕ на val ----------
def to_rank(scores):
    """Ранг у [0,1], вищий скор -> вищий ранг (для узгодженого напрямку з обома моделями)."""
    return rankdata(scores) / len(scores)

b1_val_rank, b3_val_rank = to_rank(b1_val), to_rank(b3_val)
b1_test_rank, b3_test_rank = to_rank(b1_test), to_rank(b3_test)

alpha_results = []
best_alpha, best_alpha_val_ap = None, -1
for alpha in ALPHA_GRID:
    fusion_val = alpha * b1_val_rank + (1 - alpha) * b3_val_rank
    ap = average_precision_score(y_val, fusion_val)
    roc = roc_auc_score(y_val, fusion_val)
    alpha_results.append(dict(alpha=round(alpha, 2), val_ap=ap, val_roc_auc=roc))
    if ap > best_alpha_val_ap:
        best_alpha, best_alpha_val_ap = alpha, ap

alpha_df = pd.DataFrame(alpha_results)
print("Пошук α на validation:")
display(alpha_df)

b1_ap_val_ref = average_precision_score(y_val, b1_val)
b3_ap_val_ref = average_precision_score(y_val, b3_val)
print(f"\nДовідково (validation): B1 AP={b1_ap_val_ref:.6f}, B3 AP={b3_ap_val_ref:.6f}, "
      f"найкращий fusion AP={best_alpha_val_ap:.6f} (α={best_alpha:.2f})")
print(f"✅ Обрано α={best_alpha:.2f} за validation AP={best_alpha_val_ap:.6f} (test ще не використовувався)")

Пошук α на validation:


,alpha,val_ap,val_roc_auc
0,0.0,0.009281,0.790829
1,0.1,0.007533,0.789810
2,0.2,0.007466,0.782399
3,0.3,0.007886,0.769586
4,0.4,0.008497,0.747209
5,0.5,0.008979,0.713219
6,0.6,0.010226,0.671748
7,0.7,0.011875,0.633562
8,0.8,0.014138,0.610080
9,0.9,0.021976,0.593762



Довідково (validation): B1 AP=0.075574, B3 AP=0.009281, найкращий fusion AP=0.075574 (α=1.00)
✅ Обрано α=1.00 за validation AP=0.075574 (test ще не використовувався)


In [4]:
# ---------- Фінальна оцінка на TEST — ОДИН РАЗ, з обраним α ----------
fusion_test = best_alpha * b1_test_rank + (1 - best_alpha) * b3_test_rank
ap_fusion_test = average_precision_score(y_test, fusion_test)
roc_fusion_test = roc_auc_score(y_test, fusion_test)

b1_ap_test_ref = average_precision_score(y_test, b1_test)
b1_roc_test_ref = roc_auc_score(y_test, b1_test)

print(f"B1 (референс, test):     AP={b1_ap_test_ref:.6f}  ROC-AUC={b1_roc_test_ref:.4f}")
print(f"Fusion (α={best_alpha:.2f}, test): AP={ap_fusion_test:.6f}  ROC-AUC={roc_fusion_test:.4f}")

print("\n" + "=" * 70)
if ap_fusion_test > b1_ap_test_ref:
    print(f"Fusion перевищив B1 за test AP ({ap_fusion_test:.6f} > {b1_ap_test_ref:.6f}).")
    print("Результат узгоджується з гіпотезою H4 і є підставою для переходу до Кроку 2 (B4-A/B4-B).")
else:
    print(f"Fusion НЕ перевищив B1 за test AP ({ap_fusion_test:.6f} <= {b1_ap_test_ref:.6f}).")
    print("У межах цього score-level експерименту свідчень на користь комплементарності")
    print("текстового сигналу не отримано. Це валідний негативний результат.")
print("=" * 70)

B1 (референс, test):     AP=0.018597  ROC-AUC=0.7155
Fusion (α=1.00, test): AP=0.018597  ROC-AUC=0.7155

Fusion НЕ перевищив B1 за test AP (0.018597 <= 0.018597).
У межах цього score-level експерименту свідчень на користь комплементарності
текстового сигналу не отримано. Це валідний негативний результат.


In [5]:
# ---------- Precision/Recall@K для fusion (test) — той самий формат, що й у B0-B3 ----------
def precision_recall_at_k(y_true, y_score, k_fraction):
    n = len(y_true)
    k = max(1, int(round(n * k_fraction)))
    order = np.argsort(-y_score, kind="mergesort")
    top_k_idx = order[:k]
    n_pos_in_top_k = y_true[top_k_idx].sum()
    n_pos_total = y_true.sum()
    return n_pos_in_top_k / k, (n_pos_in_top_k / n_pos_total if n_pos_total else np.nan), k

topk_rows = []
for frac in TOP_K_FRACTIONS:
    p, r, k = precision_recall_at_k(y_test, fusion_test, frac)
    topk_rows.append(dict(top_pct=f"{frac:.1%}", K=k, precision=p, recall=r))
topk_fusion_df = pd.DataFrame(topk_rows)
print("Fusion Precision/Recall@K (test):")
display(topk_fusion_df)

Fusion Precision/Recall@K (test):


,top_pct,K,precision,recall
0,0.1%,65,0.046154,0.033333
1,0.5%,326,0.042945,0.155556
2,1.0%,652,0.024540,0.177778
3,5.0%,3258,0.009515,0.344444


In [6]:
# ---------- Збереження результатів H4-перевірки ----------
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_row = dict(model="fusion_B1_B3_rank", horizon=HORIZON, best_alpha=best_alpha,
               ap_test=ap_fusion_test, roc_auc_test=roc_fusion_test,
               ap_val=best_alpha_val_ap, b1_ap_test_ref=b1_ap_test_ref,
               # Назва поля описує ЛИШЕ виміряний факт (fusion AP > B1 AP на цьому test),
               # НЕ інтерпретацію "H4 підтверджена" — це один test-порівняння, не доказ гіпотези.
               h4_test_ap_exceeds_b1=bool(ap_fusion_test > b1_ap_test_ref))
pd.DataFrame([out_row]).to_csv(RESULTS_DIR / f"step4b1_fusion_summary_h{HORIZON}.csv", index=False)
alpha_df.to_csv(RESULTS_DIR / f"step4b1_alpha_search_h{HORIZON}.csv", index=False)
topk_fusion_df.to_csv(RESULTS_DIR / f"step4b1_fusion_topk_h{HORIZON}.csv", index=False)
print(f"Збережено в {RESULTS_DIR}")

Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results
